Here is how it works:
- Full Clone and extract all config files: .yml, .yaml and related .json, .sh
- Shallow Clone from all branches: afect number commits & contributors build script or config will be only from the latest snapshop

- it sorts and index the url also save the list of random sample url indeces
- after each puase or interuption the "Cloned Repo" should be emptyied
- by a new new rerun it continues the review from the last reviewed url which is log is stored in .evn by START_NUMBER
- the sample repos are stored in "Cloned_Sample"
- this will save the sample repos as well as metrics, configs, builds and test lines
- Full Clone helps to extract full contributors and commit history
- saving metadata happend immediately so it will get lost by pause/start



In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import time, datetime
import stat

# === CONFIGURATION ===
MAX_PROJECTS = 1976
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"F:\Android_Mobile_App\AndroidProjects_FullorShallow_Full")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "8.2-Project_Metadata.csv"
config_location_csv = base_dir / "Config_Location.csv"
git_metadata_dir = base_dir / "Git_Metadata"

# === ENSURE ALL FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir, git_metadata_dir]:
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === LOAD EXISTING CONFIG LOCATIONS IF RESUMING ===
if config_location_csv.exists():
    config_locations_df = pd.read_csv(config_location_csv)
else:
    config_locations_df = pd.DataFrame(columns=["repo_name", "config_file_path", "file_type"])

# === COMMIT METADATA EXTRACTION FUNCTION ===
def extract_commit_metadata(repo_path, output_path):
    try:
        cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
        result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
        commit_hashes = result_hashes.stdout.strip().split("\n")

        rows = []
        for commit in commit_hashes:
            cmd_metadata = [
                "git", "-C", str(repo_path), "show", "--quiet",
                f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit
            ]
            result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True, check=True)
            parts = result_metadata.stdout.strip().split("|", maxsplit=4)
            if len(parts) < 5:
                continue

            cmd_files = ["git", "-C", str(repo_path), "show", "--name-only", "--pretty=format:", commit]
            result_files = subprocess.run(cmd_files, capture_output=True, text=True, check=True)
            changed_files = result_files.stdout.strip().split("\n")
            changed_files = [f.strip() for f in changed_files if f.strip()]

            count_androidTest = sum("androidTest" in f for f in changed_files)
            count_github_workflows = sum(".github/workflows" in f for f in changed_files)
            count_gradle = sum("build.gradle" in f for f in changed_files)

            rows.append({
                "commit_hash": parts[0],
                "author_name": parts[1],
                "author_email": parts[2],
                "commit_date": parts[3],
                "commit_message": parts[4],
                "touches_androidTest": count_androidTest > 0,
                "count_androidTest": count_androidTest,
                "touches_github_workflows": count_github_workflows > 0,
                "count_github_workflows": count_github_workflows,
                "touches_gradle": count_gradle > 0,
                "count_gradle": count_gradle
            })

        if rows:
            df = pd.DataFrame(rows)
            output_path.mkdir(parents=True, exist_ok=True)
            df.to_csv(output_path / "contributors_commits.csv", index=False)
            print(f"✅ Saved commit metadata for {repo_path.name}")
        else:
            print(f"⚠️ No commit data for {repo_path.name}")

    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to extract commit data for {repo_path.name}: {e}")

# === FUNCTION TO HANDLE READ-ONLY FILES ===
def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

# === FUNCTION TO GET COUNT FROM GITHUB API ===
def get_count(api_url, headers):
    try:
        r = requests.get(api_url, headers=headers, params={"per_page": 1})
        if 'Link' in r.headers:
            return int(r.headers['Link'].split(',')[0].split('page=')[-1].split('>')[0])
        return len(r.json())
    except:
        return 0

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.loc[i, 'github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', url, str(repo_path)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        #subprocess.run(['git', 'clone', '--depth', '1', '--no-single-branch', url, str(repo_path)],stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) #shallow Clone
        print("✅ Clone complete")
    except Exception as e:
        print(f"❌ Clone failed for {repo_name}: {e}")
        continue

    ci_keywords = ['ci', 'build', 'test', 'workflow', 'pipeline', 'instrumentation']
    config_files_found = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            if file_lower.endswith(('.yml', '.yaml')):
                config_files_found.append({
                    "repo_name": repo_name,
                    "config_file_path": rel_path,
                    "file_type": file_lower.split('.')[-1]
                })
            elif file_lower.endswith(('.json', '.sh')):
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read().lower()
                        if any(keyword in content for keyword in ci_keywords):
                            config_files_found.append({
                                "repo_name": repo_name,
                                "config_file_path": rel_path,
                                "file_type": file_lower.split('.')[-1]
                            })
                except Exception as e:
                    print(f"⚠️ Could not read {rel_path} in {repo_name}: {e}")

    if config_files_found:
        config_locations_df = pd.concat([config_locations_df, pd.DataFrame(config_files_found)], ignore_index=True)

    # === Extract commit metadata and save ===
    metadata_dir = git_metadata_dir / repo_name
    extract_commit_metadata(repo_path, metadata_dir)

    # === Fetch and save metadata ===
    try:
        headers = {'Authorization': f'token {GITHUB_TOKEN}'}
        base_api = f"https://api.github.com/repos/{username}/{project}"
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json()

        metadata_row = {
            'project_name': project,
            'repo_name': repo_name,
            'full_name': data.get('full_name'),
            'description': data.get('description'),
            'language': data.get('language'),
            'license': data.get('license', {}).get('name') if data.get('license') else None,
            'created_at': data.get('created_at'),
            'updated_at': data.get('updated_at'),
            'last_commit_date': data.get('pushed_at'),
            'stars': data.get('stargazers_count'),
            'forks': data.get('forks_count'),
            'watchers': data.get('watchers_count'),
            'open_issues': data.get('open_issues_count'),
            'contributors': get_count(f"{base_api}/contributors", headers),
            'pull_requests': get_count(f"{base_api}/pulls?state=all", headers),
            'commits': get_count(f"{base_api}/commits", headers),
            'size': data.get('size')
        }

        metadata_df = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            metadata_df.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            metadata_df.to_csv(metadata_path, mode='w', header=True, index=False)

        print("🧾 Metadata saved")

    except Exception as e:
        print(f"⚠️ Metadata error for {repo_name}: {e}")

    # === Move to Cloned_Sample or delete ===
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📦 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, onerror=force_remove_readonly)
            print(f"🧹 Deleted cloned repo: {repo_name}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))
    config_locations_df.drop_duplicates().to_csv(config_location_csv, index=False)

print(f"\n✅ Process complete. Sampled: {len(sample_indices_to_keep)} | Total Processed: {len(df) - (START_NUMBER - 1)}")
print("\n✅ All selected repositories have been processed.")


🔁 Loaded SAMPLE_LIST from .env with 150 indices.

🔍 [7/1796] Processing 0006.slartus.4pdaClient-plus...
